In [ ]:
!pip install flask pyngrok openai anthropic ffmpeg-python opencv-python-headless deepface --quiet

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
from openai import OpenAI
import anthropic
import subprocess
import os
import time
import cv2
import base64
import numpy as np
from deepface import DeepFace

# 🔐 Keys
NGROK_TOKEN       = ""
OPENAI_API_KEY    = ""
ANTHROPIC_API_KEY = ""

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
openai_client  = OpenAI()
claude_client  = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

app = Flask(__name__)
conversation_history = []

# โหลด face detector ครั้งเดียวตอนเริ่ม
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Warm up DeepFace
print("⏳ Warming up DeepFace...")
try:
    DeepFace.analyze(np.zeros((224, 224, 3), dtype=np.uint8), actions=['emotion'], enforce_detection=False)
except:
    pass
print("✅ DeepFace ready!")

EMOTION_MAP_TH = {
    "angry":     "โกรธ",
    "disgust":   "รังเกียจ",
    "fear":      "กลัว",
    "happy":     "มีความสุข",
    "sad":       "เศร้า",
    "surprise":  "ตกใจ",
    "neutral":   "เฉยๆ"
}

def crop_face(img_bytes):
    """crop ใบหน้าที่ใหญ่ที่สุด พร้อม padding 30% — คืน (face_bytes, face_img_bgr)"""
    npimg = np.frombuffer(img_bytes, np.uint8)
    img   = cv2.imdecode(npimg, cv2.IMREAD_COLOR)
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

    if len(faces) == 0:
        print("⚠️ หาใบหน้าไม่เจอ ใช้ภาพต้นฉบับ")
        _, buf = cv2.imencode('.jpg', img)
        return buf.tobytes(), img

    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    pad = int(max(w, h) * 0.3)
    x1  = max(0, x - pad)
    y1  = max(0, y - pad)
    x2  = min(img.shape[1], x + w + pad)
    y2  = min(img.shape[0], y + h + pad)

    face_img = img[y1:y2, x1:x2]
    _, buf = cv2.imencode('.jpg', face_img)
    print(f"✅ crop ใบหน้าสำเร็จ ({w}x{h}px)")
    return buf.tobytes(), face_img


def analyze_face_emotion(face_img_bgr):
    """วิเคราะห์ emotion จาก face img — คืนชื่ออารมณ์ภาษาไทย"""
    try:
        result = DeepFace.analyze(face_img_bgr, actions=['emotion'], enforce_detection=False)
        top_en = result[0]['dominant_emotion'] if isinstance(result, list) else result['dominant_emotion']
        top_th = EMOTION_MAP_TH.get(top_en.lower(), top_en)
        print(f"😊 Face emotion: {top_th} ({top_en})")
        return top_th
    except Exception as e:
        print(f"⚠️ DeepFace failed: {e}")
        return None


@app.route('/analyze', methods=['POST'])
def analyze():
    global conversation_history

    try:
        # ---- รับและบันทึกเสียง ----
        audio_file = request.files["audio"]
        audio_file.save("input_raw.wav")

        subprocess.run([
            "ffmpeg", "-y",
            "-i", "input_raw.wav",
            "-ar", "16000", "-ac", "1",
            "-af", "highpass=f=200,lowpass=f=3000,afftdn=nf=-25",
            "input.wav"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # ---- Whisper แปลงเสียง ----
        t1 = time.time()
        with open("input.wav", "rb") as f:
            transcript = openai_client.audio.transcriptions.create(
                model="whisper-1", file=f, language="th"
            )
        text = transcript.text.strip()
        print(f"🎤 Whisper: {time.time()-t1:.2f}s | {text}")

        if len(text) < 2:
            return jsonify({"response": ""})

        # ---- รับภาพ → crop ใบหน้า → วิเคราะห์ emotion → base64 ----
        b64_frame    = None
        face_emotion = None

        if "image" in request.files:
            img_bytes            = request.files["image"].read()
            face_bytes, face_img = crop_face(img_bytes)
            face_emotion         = analyze_face_emotion(face_img)   # ← วิเคราะห์ตรงนี้
            b64_frame            = base64.b64encode(face_bytes).decode("utf-8")

        # ---- สร้าง text ที่แนบ emotion ใบหน้าด้วย ----
        if face_emotion:
            enriched_text = f"[อารมณ์จากใบหน้า: {face_emotion}] {text}"
        else:
            enriched_text = text

        # ---- สร้าง content — ข้อความก่อน ภาพทีหลัง ----
        content = [{"type": "text", "text": enriched_text}]
        if b64_frame:
            content.append({
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/jpeg",
                    "data": b64_frame
                }
            })

        conversation_history.append({"role": "user", "content": content})
        conversation_history = conversation_history[-6:]

        # ---- Claude ตอบ ----
        t2 = time.time()
        response = claude_client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=300,
            system=(
                 "คุณคือกะปิที่มีบุคลิกเหมือนมนุษย์ พูดภาษาไทยเท่านั้น "
    "ตอบโดยยึดเนื้อหาจากเสียงที่พูดเป็นหลักเสมอ "
    "ข้อมูลในวงเล็บ [อารมณ์จากใบหน้า: ...] คือผลวิเคราะห์สีหน้าจริง "
    "ถ้าสีหน้ากับเนื้อหาที่พูด 'สอดคล้องกัน' ให้ตอบตามปกติ "
    "ถ้าสีหน้ากับเนื้อหาที่พูด 'ไม่สอดคล้องกัน' เช่น พูดเรื่องดีแต่หน้าเศร้า "
    "ให้ตอบรับเนื้อหาที่พูดก่อน แล้วค่อยพูดถึงสีหน้าที่เห็นด้วยความห่วงใย "
    "เช่น 'อากาศดีจริงๆ เลย แต่สีหน้าดูไม่ค่อยสดใสเลยนะ เป็นอะไรอยู่หรือเปล่า?' "
    "ห้ามพูดถึงสิ่งของ พื้นหลัง หรือสภาพแวดล้อมในภาพเด็ดขาด "
    "ตอบตรงประเด็น ไม่อ้อมค้อม มีความเห็นอกเห็นใจ รับฟังอย่างตั้งใจ "
    "ให้กำลังใจเมื่อผู้พูดเครียดหรือเศร้า "
    "ตอบสั้นกระชับ ไม่เกิน 3-4 ประโยค เป็นธรรมชาติเหมือนคุยกับเพื่อน "
    "ถ้าข้อความไม่ชัดเจนหรือสะกดผิด ให้เดาความหมายจากบริบทแล้วตอบเลย "
    "ห้ามทวนคำถามหรือประโยคที่ผู้พูดพูดมาเด็ดขาด"
    "ถ้าฟังไม่รู้เรื่องให้ตอบว่า กะปิฟังไม่ออก พูดใหม่อีกครั้ง"
            ),
            messages=conversation_history
        )

        reply = response.content[0].text
        print(f"🤖 Claude: {time.time()-t2:.2f}s | 🐻 {reply}")

        # เก็บ history เฉพาะ text
        conversation_history[-1] = {"role": "user", "content": enriched_text}
        conversation_history.append({"role": "assistant", "content": reply})

        return jsonify({
            "response":     reply,
            "face_emotion": face_emotion or "ไม่พบใบหน้า"
        })

    except Exception as e:
        print("❌ ERROR:", e)
        return jsonify({"response": "เกิดข้อผิดพลาดภายในระบบ"}), 500


ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000)
print("=========================================")
print(f"📌 URL สำหรับ Raspberry Pi: {public_url.public_url}/analyze")
print("=========================================")

app.run(host="0.0.0.0", port=5000)